# 06 — Deep Learning Model: Keras MLP

This notebook builds and trains the Keras MLP model using `KerasClassifierWrapper`, visualizes training history (loss/accuracy curves), and evaluates on the test set.

**Architecture:**
- Dense(256) -> BatchNorm -> Dropout(0.3)
- Dense(128) -> BatchNorm -> Dropout(0.3)
- Dense(64)  -> BatchNorm -> Dropout(0.3)
- Dense(1, sigmoid)

**Callbacks:** EarlyStopping, ReduceLROnPlateau

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import keras

from src.config import DATASETS, SEED, DL_PARAMS
from src.data.loader import load_raw_dataset, get_target_column
from src.data.preprocessor import MediSensePreprocessor
from src.data.splitter import stratified_split
from src.models.dl_model import build_dl_model, KerasClassifierWrapper
from src.evaluation.metrics import compute_metrics, format_metrics

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

np.random.seed(SEED)

## 1. Prepare Data (Heart Disease as Primary Example)

In [ ]:
dataset_name = 'heart'
df = load_raw_dataset(dataset_name)
target_col = get_target_column(dataset_name)

# Clean
df['ca'] = pd.to_numeric(df['ca'], errors='coerce')
df['thal'] = pd.to_numeric(df['thal'], errors='coerce')
df['target'] = (df['target'] > 0).astype(int)

# Split
train_df, val_df, test_df = stratified_split(df, target_col)

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col].values
X_val = val_df.drop(columns=[target_col])
y_val = val_df[target_col].values
X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col].values

# Preprocess
preprocessor = MediSensePreprocessor(dataset_name)
X_train_proc = preprocessor.fit_transform(X_train, y_train)
X_val_proc = preprocessor.transform(X_val)
X_test_proc = preprocessor.transform(X_test)

print(f"Train: {X_train_proc.shape}, Val: {X_val_proc.shape}, Test: {X_test_proc.shape}")

## 2. Build and Inspect Model Architecture

In [ ]:
input_dim = X_train_proc.shape[1]
model = build_dl_model(input_dim)
model.summary()

## 3. Train the Model

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=DL_PARAMS['patience'],
        restore_best_weights=True,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5
    ),
]

history = model.fit(
    X_train_proc, y_train,
    epochs=DL_PARAMS['max_epochs'],
    batch_size=DL_PARAMS['batch_size'],
    validation_data=(X_val_proc, y_val),
    callbacks=callbacks,
    verbose=1,
)

print(f"\nTraining stopped at epoch {len(history.history['loss'])}")

## 4. Training History: Loss and Accuracy Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Binary Crossentropy Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

# Accuracy
axes[1].plot(history.history['accuracy'], label='Train Accuracy')
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

# AUC
axes[2].plot(history.history['auc'], label='Train AUC')
axes[2].plot(history.history['val_auc'], label='Val AUC')
axes[2].set_title('AUC')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('AUC')
axes[2].legend()

plt.suptitle('Training History — Keras MLP (Heart Disease)', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Evaluate on Test Set

In [ ]:
y_prob = model.predict(X_test_proc, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

metrics = compute_metrics(y_test, y_pred, y_prob)
print("Test Set Metrics (Heart Disease):")
print(format_metrics(metrics))

## 6. Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['No Disease', 'Disease'])
disp.plot(ax=ax, cmap='Blues')
ax.set_title('Confusion Matrix — Keras MLP (Heart Disease)')
plt.tight_layout()
plt.show()

## 7. Train on All Three Datasets Using KerasClassifierWrapper

In [ ]:
for ds_name in ['heart', 'diabetes', 'liver']:
    print(f"\n{'=' * 50}")
    print(f"Dataset: {ds_name.upper()}")
    print(f"{'=' * 50}")
    
    df = load_raw_dataset(ds_name)
    target_col = get_target_column(ds_name)
    
    if ds_name == 'heart':
        df['ca'] = pd.to_numeric(df['ca'], errors='coerce')
        df['thal'] = pd.to_numeric(df['thal'], errors='coerce')
        df['target'] = (df['target'] > 0).astype(int)
    elif ds_name == 'liver':
        df['Dataset'] = df['Dataset'].map({1: 1, 2: 0})
    
    train_df, val_df, test_df = stratified_split(df, target_col)
    X_tr = train_df.drop(columns=[target_col])
    y_tr = train_df[target_col].values
    X_te = test_df.drop(columns=[target_col])
    y_te = test_df[target_col].values
    
    prep = MediSensePreprocessor(ds_name)
    X_tr_p = prep.fit_transform(X_tr, y_tr)
    X_te_p = prep.transform(X_te)
    
    wrapper = KerasClassifierWrapper(input_dim=X_tr_p.shape[1])
    wrapper.fit(X_tr_p, y_tr)
    
    y_pred = wrapper.predict(X_te_p)
    y_prob = wrapper.predict_proba(X_te_p)[:, 1]
    m = compute_metrics(y_te, y_pred, y_prob)
    print(format_metrics(m))

## Summary

- The Keras MLP with BatchNorm and Dropout achieves competitive performance.
- EarlyStopping prevents overfitting by restoring best weights.
- Training curves show good convergence on the heart disease dataset.
- The `KerasClassifierWrapper` provides a scikit-learn compatible interface for integration with the stacking ensemble.